YouTube Yorum NLP Pipeline
===========================
AŞAMA 1 : Keşifçi Veri Analizi (EDA)

AŞAMA 2 : Veri Ön İşleme (Preprocessing)

AŞAMA 3 : Veri Formatlama ve Bölme

AŞAMA 4 : Tokenization


**Giriş:** * `etiketlenmis_yorumlar.csv` *(etiketleme scripti çıktısı)*

**Çıktılar:**
* `temiz_veri.csv` *(model eğitimine hazır ham metin + etiket)*
* `train.csv` / `val.csv` / `test.csv` *(bölünmüş veri)*
* `tokenized_train.pt` *(PyTorch tensörleri)*
* `eda_rapor/` *(grafikler)*

In [1]:
import os
import re
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")           # sunucusuz ortamda grafik kaydetmek için
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from transformers import AutoTokenizer
from tqdm import tqdm
import emoji

warnings.filterwarnings("ignore")

In [2]:
INPUT_FILE      = "anaveri.csv"   # etiketlenmiş veri
OUTPUT_DIR      = "eda_rapor"                   # grafik klasörü
TEMIZ_DOSYA     = "temiz_veri.csv"
TRAIN_DOSYA     = "train.csv"
VAL_DOSYA       = "val.csv"
TEST_DOSYA      = "test.csv"
TOKEN_DOSYA     = "tokenized.pt"

TEXT_COL        = "Yorum Metni"
LABEL_COL       = "Nihai_Etiket"               # etiket sütunu
MODEL_ADI       = "xlm-roberta-base"  # XML ROBERT tabanlı model
MAX_LENGTH      = 128
MIN_KELIME      = 2                             # bu altı kelime → sil
TRAIN_ORAN      = 0.80
VAL_ORAN        = 0.10
TEST_ORAN       = 0.10

ETIKET_MAP = {
    "olumlu"        : 0,
    "olumsuz"       : 1,
    "nötr"          : 2,
    "öneri/tavsiye" : 3,
    "şikayet"       : 4,
}

In [3]:
print("=" * 60)
print("YouTube Yorum NLP Pipeline başlatıldı")
print("=" * 60)

YouTube Yorum NLP Pipeline başlatıldı


## [AŞAMA 1] Keşifçi Veri Analizi (EDA)

In [4]:
df = pd.read_csv(INPUT_FILE)
print(f"  Yüklenen satır sayısı : {len(df)}")
print(f"  Kolonlar              : {df.columns.tolist()}")


  Yüklenen satır sayısı : 7609
  Kolonlar              : ['Video Başlığı', 'Video Kaynağı', 'Yorum ID', 'Üst Yorum ID', 'Yorum Yapan Kişi', 'Yanıtlanan Kişi', 'Yorum Metni', 'Beğeni Sayısı', 'Tarih', 'Yanıt Mı?', 'XLM_Roberta_Etiketi', 'XLM_Roberta_Confidence', 'Kural_Etiketi', 'XLM_Turkce', 'Nihai_Etiket']


In [5]:
print(f"\n  [1.1] Temel istatistikler")
print(f"  Toplam yorum          : {len(df)}")
print(f"  Benzersiz video       : {df['Video Başlığı'].nunique()}")
print(f"  Toplam null (metin)   : {df[TEXT_COL].isnull().sum()}")


  [1.1] Temel istatistikler
  Toplam yorum          : 7609
  Benzersiz video       : 6
  Toplam null (metin)   : 0


In [6]:
# Boş metinleri doldur
df[TEXT_COL] = df[TEXT_COL].fillna("").str.strip()

In [7]:
print(f"\n  [1.2] Spam ve kopya temizliği")
onceki = len(df)
df = df.drop_duplicates(subset=[TEXT_COL])     # aynı yorum metni → tek tut
df = df.reset_index(drop=True)
print(f"  Silinen kopya yorum   : {onceki - len(df)}")
print(f"  Kalan yorum           : {len(df)}")


  [1.2] Spam ve kopya temizliği
  Silinen kopya yorum   : 117
  Kalan yorum           : 7492


In [8]:
# ── 1.3 METİN UZUNLUĞU ANALİZİ ──────────────────────────────
print(f"\n  [1.3] Metin uzunluğu analizi")
df["kelime_sayisi"] = df[TEXT_COL].apply(lambda x: len(str(x).split()))
df["karakter_sayisi"] = df[TEXT_COL].apply(len)


print(f"  Ortalama kelime sayısı: {df['kelime_sayisi'].mean():.1f}")
print(f"  Medyan kelime sayısı  : {df['kelime_sayisi'].median():.1f}")
print(f"  Max kelime sayısı     : {df['kelime_sayisi'].max()}")
print(f"  Min kelime sayısı     : {df['kelime_sayisi'].min()}")



  [1.3] Metin uzunluğu analizi
  Ortalama kelime sayısı: 11.7
  Medyan kelime sayısı  : 7.0
  Max kelime sayısı     : 620
  Min kelime sayısı     : 1


In [9]:
os.makedirs("eda_rapor", exist_ok=True)

In [10]:
# Histogram: kelime sayısı dağılımı
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["kelime_sayisi"].clip(upper=100), bins=50, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Kelime Sayısı")
ax.set_ylabel("Yorum Adedi")
ax.set_title("Yorum Kelime Sayısı Dağılımı")
ax.axvline(MIN_KELIME, color="red",    linestyle="--", label=f"Min eşik ({MIN_KELIME})")
ax.axvline(MAX_LENGTH, color="orange", linestyle="--", label=f"Max token ({MAX_LENGTH})")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "kelime_dagilimi.png"), dpi=150)
plt.close(fig)
print(f"  Grafik kaydedildi: {OUTPUT_DIR}/kelime_dagilimi.png")

  Grafik kaydedildi: eda_rapor/kelime_dagilimi.png


In [11]:
# Çok kısa yorumları sil
onceki = len(df)
df = df[df["kelime_sayisi"] >= MIN_KELIME].reset_index(drop=True)
print(f"  {MIN_KELIME} kelimeden kısa silinecek : {onceki - len(df)}")
print(f"  Kalan yorum            : {len(df)}")

  2 kelimeden kısa silinecek : 274
  Kalan yorum            : 7218


In [12]:
# ── 1.4 SINIF DAĞILIMI (CLASS IMBALANCE) ─────────────────────
print(f"\n  [1.4] Sınıf dağılımı kontrolü")
sinif_dagilimi = df[LABEL_COL].value_counts()
print(sinif_dagilimi.to_string())


  [1.4] Sınıf dağılımı kontrolü
Nihai_Etiket
olumlu           3366
olumsuz          1985
nötr             1444
öneri/tavsiye     379
şikayet            44


In [13]:
fig, ax = plt.subplots(figsize=(8, 4))
renk_paleti = ["#2ECC71", "#E74C3C", "#95A5A6", "#3498DB", "#E67E22"]
sinif_dagilimi.plot(kind="bar", ax=ax, color=renk_paleti, edgecolor="white")
ax.set_title("Sınıf Dağılımı (Etiket Frekansları)")
ax.set_xlabel("Etiket")
ax.set_ylabel("Yorum Adedi")
ax.tick_params(axis="x", rotation=30)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "sinif_dagilimi.png"), dpi=150)
plt.close(fig)
print(f"  Grafik kaydedildi: {OUTPUT_DIR}/sinif_dagilimi.png")

  Grafik kaydedildi: eda_rapor/sinif_dagilimi.png


In [14]:
# Dengesizlik uyarısı
en_fazla = sinif_dagilimi.max()
en_az    = sinif_dagilimi.min()
oran     = en_fazla / en_az
if oran > 3:
    print(f"  ⚠️  Sınıf dengesizliği yüksek (oran: {oran:.1f}x) → eğitimde class_weight kullan!")

  ⚠️  Sınıf dengesizliği yüksek (oran: 76.5x) → eğitimde class_weight kullan!


In [15]:
# ── 1.5 N-GRAM ANALİZİ ───────────────────────────────────────
print(f"\n  [1.5] N-gram analizi (şikayet ve öneri sınıfları)")

def ngram_frekans(metin_listesi, n=2, top_k=15):
    """n'li kelime gruplarının frekansını döndürür."""
    tum_ngramlar = []
    for metin in metin_listesi:
        kelimeler = str(metin).lower().split()
        tum_ngramlar.extend([" ".join(kelimeler[i:i+n]) for i in range(len(kelimeler)-n+1)])
    return Counter(tum_ngramlar).most_common(top_k)


  [1.5] N-gram analizi (şikayet ve öneri sınıfları)


In [16]:
for hedef_sinif in ["şikayet", "öneri/tavsiye"]:
    if hedef_sinif not in df[LABEL_COL].values:
        continue
    alt_df   = df[df[LABEL_COL] == hedef_sinif][TEXT_COL].tolist()
    bigramlar = ngram_frekans(alt_df, n=2, top_k=15)
    trigramlar = ngram_frekans(alt_df, n=3, top_k=10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    # bigram
    bk, bv = zip(*bigramlar) if bigramlar else ([], [])
    axes[0].barh(list(bk)[::-1], list(bv)[::-1], color="#3498DB")
    axes[0].set_title(f"[{hedef_sinif}] En Sık 2-gram")
    # trigram
    tk, tv = zip(*trigramlar) if trigramlar else ([], [])
    axes[1].barh(list(tk)[::-1], list(tv)[::-1], color="#E74C3C")
    axes[1].set_title(f"[{hedef_sinif}] En Sık 3-gram")
    fig.tight_layout()
    dosya_adi = hedef_sinif.replace("/", "_")
    fig.savefig(os.path.join(OUTPUT_DIR, f"ngram_{dosya_adi}.png"), dpi=150)
    plt.close(fig)
    print(f"  Grafik kaydedildi: {OUTPUT_DIR}/ngram_{dosya_adi}.png")
    print(f"  [{hedef_sinif}] Top-5 bigram: {bigramlar[:5]}")

  Grafik kaydedildi: eda_rapor/ngram_şikayet.png
  [şikayet] Top-5 bigram: [('çok kötü', 11), ('cem yılmaz', 7), ('kötü bir', 4), ('taner abi', 4), ('taner abiye', 3)]
  Grafik kaydedildi: eda_rapor/ngram_öneri_tavsiye.png
  [öneri/tavsiye] Top-5 bigram: [('sezon gelsin', 30), ('2. sezon', 25), ('devamı gelsin', 17), ('gelsin lütfen', 14), ('daha çok', 13)]


## AŞAMA 2 : VERİ ÖN İŞLEME (PREPROCESSING)

In [17]:
print("  [2.1] URL ve e-posta temizliği")
URL_RE   = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
print("silme işleminden önceki yorum sayısı : ", len(df))
def url_temizle(metin):
    metin = URL_RE.sub("", metin)
    metin = EMAIL_RE.sub("", metin)
    return metin

df[TEXT_COL] = df[TEXT_COL].apply(url_temizle)
print("silme işleminden sonraki yorum sayısı : ", len(df))


  [2.1] URL ve e-posta temizliği
silme işleminden önceki yorum sayısı :  7218
silme işleminden sonraki yorum sayısı :  7218


In [18]:
print("  [2.2] Zaman damgası ve @etiket dönüşümü")
ZAMAN_RE = re.compile(r"\b\d{1,2}:\d{2}(:\d{2})?\b")  # 15:23 veya 1:05:30
print("silme işleminden önceki yorum sayısı : ", len(df))

def zaman_ve_etiket_temizle(metin):
    # Zaman damgasını tamamen sil
    metin = ZAMAN_RE.sub("", metin)
    # @kullanici → standart <kullaniciadi> token'ı
    metin = re.sub(r"@\w+", "<kullaniciadi>", metin)
    return metin

df[TEXT_COL] = df[TEXT_COL].apply(zaman_ve_etiket_temizle)
print("silme işleminden sonraki yorum sayısı : ", len(df))


  [2.2] Zaman damgası ve @etiket dönüşümü
silme işleminden önceki yorum sayısı :  7218
silme işleminden sonraki yorum sayısı :  7218


Seçilmiş olan XML ROBERT modeli emojilere duyarlı olduğu için emojileri metne dönüştürme işlemine gerek yoktur ama diğer modeller için şu işlem uygulanabilir 

In [19]:
# ── 2.4 GEREKSIZ KARAKTER TEMİZLİĞİ ─────────────────────────
print("  [2.4] Gereksiz karakter temizliği (!, ? ve emojiler KORUNUYOR)")
print("silme işleminden önceki yorum sayısı : ", len(df))

def karakter_temizle(metin):
    # Sadece silinmesini istediğimiz gereksiz noktalama işaretlerini belirtiyoruz.
    # Böylece harfler, rakamlar, ! ? ve EMOJİLER güvende kalıyor.
    metin = re.sub(r'[\.\,\;\:\(\)\{\}\"\'\-\_\|\*\&\^\%\$\#\~\`\/\\]', ' ', metin)
    
    # Çoklu boşlukları tek boşluğa indir
    metin = re.sub(r" +", " ", metin).strip()
    return metin

df[TEXT_COL] = df[TEXT_COL].apply(karakter_temizle)
print("silme işleminden sonraki yorum sayısı : ", len(df))

  [2.4] Gereksiz karakter temizliği (!, ? ve emojiler KORUNUYOR)
silme işleminden önceki yorum sayısı :  7218
silme işleminden sonraki yorum sayısı :  7218


DUYGU YOĞUNLUĞU KORUMA'nın amacı yorum netinlerinde ki süperr berbatt veya daha nötr olan ablaa gibi yorumların barındırdığı heyecan şikayet ve mennuniyet hissiyatını modele doğru bir şekilde aktarmak. Seçilen model Cased olduğu için Büyük ve küçük harf ayrımını yapabilmektedir. Bu da bize duygu yoğunluğunu doğru bir şekilde ifade etme de avantaj sağlar. 

In [20]:
# --- [2.5] UZATILMIŞ HARF NORMALİZASYONU VE VURGU KORUMA ---
print("silme işleminden önceki yorum sayısı : ", len(df))

def uzatma_ve_vurgu_normalize(metin):
    metin = str(metin)
    # Kelime kelime bölerek işlem yapıyoruz
    kelimeler = metin.split()
    yeni_kelimeler = []
    
    for kelime in kelimeler:
        # Eğer kelimede 3+ tekrar eden harf varsa (Örn: berbaaaat, lütfennn)
        if re.search(r"(.)\1{2,}", kelime):
            # 1. Harf tekrarını 2'ye indir (berbaat, lütfenn)
            temiz_kelime = re.sub(r"(.)\1{2,}", r"\1\1", kelime)
            # 2. Vurguyu belirtmek için kelimeyi tamamen BÜYÜK yap
            yeni_kelimeler.append(temiz_kelime.upper())
        else:
            # Uzatma yoksa kelimeye dokunma (küçük/büyük hali korunsun)
            yeni_kelimeler.append(kelime)
            
    return " ".join(yeni_kelimeler)

print("\n[AŞAMA 2.5] Uzatılmış kelimeler normalize ediliyor ve BÜYÜTÜLÜYOR...")

# Fonksiyonu tüm veri setine (DataFrame) uyguluyoruz
df[TEXT_COL] = df[TEXT_COL].apply(uzatma_ve_vurgu_normalize)

# Temizlik sonrası boş kalan veya sadece boşluktan oluşan satırları temizle
df[TEXT_COL] = df[TEXT_COL].str.strip()
df = df[df[TEXT_COL] != ""].reset_index(drop=True)
print("silme işleminden sonraki yorum sayısı : ", len(df))

silme işleminden önceki yorum sayısı :  7218

[AŞAMA 2.5] Uzatılmış kelimeler normalize ediliyor ve BÜYÜTÜLÜYOR...
silme işleminden sonraki yorum sayısı :  7212


In [21]:
# 1. Metinlerin asla kesilmemesi için ayarlar
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

def gercek_vurgu_var_mi(metin):
    metin = str(metin)
    # Önce bizim koyduğumuz <USER> etiketini geçici olarak temizleyelim ki kuralı bozmasın
    gecici_metin = metin.replace("<USER>", "")
    
    # Şimdi kalan metinde en az 3 harfli TAMAMI BÜYÜK bir kelime var mı bakalım
    if re.search(r"\b[A-ZÇĞİÖŞÜ]{3,}\b", gecici_metin):
        return True
    return False

# Filtreleme yapıyoruz
gercek_vurgu_df = df[df[TEXT_COL].apply(gercek_vurgu_var_mi)]

print(f"Toplam gerçek vurgulu yorum sayısı: {len(gercek_vurgu_df)}")
print("-" * 60)

# Örnekleri tam metin olarak listeleme
# head(50) diyerek ilk 50 tanesini tam görebilirsin
for i, satir in enumerate(gercek_vurgu_df[TEXT_COL].head(50)):
    print(f"Yorum {i+1}:\n{satir}\n{'-'*30}")

Toplam gerçek vurgulu yorum sayısı: 1274
------------------------------------------------------------
Yorum 1:
AÇIK MUTFAK ÇOCUKLUK TRAVMALARIM
------------------------------
Yorum 2:
Tuğba bir daha gelsin LÜTFENN ikinci izleyişim bundada ikinci olmasına rağmennyine gülmekten ağladım😂
------------------------------
Yorum 3:
AYY Tuğba neler yaşamışsın YAA ne kadar güldümm iyi ki geldin 😅🫶🏻
------------------------------
Yorum 4:
YAA çok iyi ve güzel din😂
------------------------------
Yorum 5:
ABLA BENDE GAZIANTEPLIYIMMN
------------------------------
Yorum 6:
Niye bu kadar TATLISINN
------------------------------
Yorum 7:
Çok tatlısın ABLAA❤❤
------------------------------
Yorum 8:
SKSKSKSKSSK ANIRDIM
------------------------------
Yorum 9:
Tuğba ABLAA MERHABAA FANINIMM tanışalım mı kalp at NOLURR❤❤😊😊❤❤🎉🎉
------------------------------
Yorum 10:
Ben DEE Bu kız bir şey sunmalı stand up fln bir şey yapmalı Çok geç fark etmişiz yaa Doğalının daha komik olması efsane
----------------------

## AŞAMA 3 : VERİ FORMATLAMA VE BÖLME

In [22]:
# ── 3.1 ETİKET SAYISALLAŞTIRMA ────────────────────────────────
print("  [3.1] Etiket sayısallaştırma")

# Etiket haritası üzerinden sayısallaştır
# Bilinmeyen etiket varsa NaN olur → düşür
df["etiket_id"] = df[LABEL_COL].map(ETIKET_MAP)

bilinmeyen = df["etiket_id"].isnull().sum()
if bilinmeyen > 0:
    print(f"  ⚠️  Bilinmeyen etiket bulundu: {bilinmeyen} → siliniyor")
    df = df[df["etiket_id"].notna()].reset_index(drop=True)

df["etiket_id"] = df["etiket_id"].astype(int)

print(f"  Etiket haritası: {ETIKET_MAP}")
print(f"  Sayısallaştırma sonrası yorum: {len(df)}")

  [3.1] Etiket sayısallaştırma
  Etiket haritası: {'olumlu': 0, 'olumsuz': 1, 'nötr': 2, 'öneri/tavsiye': 3, 'şikayet': 4}
  Sayısallaştırma sonrası yorum: 7212


In [23]:
print("  [3.2] Model için gerekli sütunlar hazırlanıyor")

# Model için sadece metin + etiket tutuyoruz; diğer kolonları da saklıyoruz
sutunlar_tut = [
    "Video Başlığı", "Yorum ID", TEXT_COL, LABEL_COL,
    "etiket_id", "Beğeni Sayısı", "Yanıt Mı?",
    "XLM_Roberta_Etiketi", "XLM_Roberta_Confidence",
    "Kural_Etiketi",
]
sutunlar_tut = [s for s in sutunlar_tut if s in df.columns]
df_temiz = df[sutunlar_tut].copy()

df_temiz.to_csv(TEMIZ_DOSYA, index=False, encoding="utf-8-sig")
print(f"  Temiz veri kaydedildi: {TEMIZ_DOSYA}")

  [3.2] Model için gerekli sütunlar hazırlanıyor
  Temiz veri kaydedildi: temiz_veri.csv


In [24]:
# ── 3.3 TRAIN / VAL / TEST BÖLME (%80 / %10 / %10) ───────────
print(f"  [3.3] Veri bölme: Train {TRAIN_ORAN*100:.0f}% / "
      f"Val {VAL_ORAN*100:.0f}% / Test {TEST_ORAN*100:.0f}%")

# Önce train vs (val+test), sonra val vs test
df_train, df_val_test = train_test_split(
    df_temiz,
    test_size=(VAL_ORAN + TEST_ORAN),
    random_state=42,
    stratify=df_temiz["etiket_id"],   # sınıf oranını koru
)

val_test_oran = TEST_ORAN / (VAL_ORAN + TEST_ORAN)
df_val, df_test = train_test_split(
    df_val_test,
    test_size=val_test_oran,
    random_state=42,
    stratify=df_val_test["etiket_id"],
)

df_train.to_csv(TRAIN_DOSYA, index=False, encoding="utf-8-sig")
df_val.to_csv(VAL_DOSYA,   index=False, encoding="utf-8-sig")
df_test.to_csv(TEST_DOSYA,  index=False, encoding="utf-8-sig")

print(f"  Train : {len(df_train)} yorum → {TRAIN_DOSYA}")
print(f"  Val   : {len(df_val)}   yorum → {VAL_DOSYA}")
print(f"  Test  : {len(df_test)}  yorum → {TEST_DOSYA}")

  [3.3] Veri bölme: Train 80% / Val 10% / Test 10%
  Train : 5769 yorum → train.csv
  Val   : 721   yorum → val.csv
  Test  : 722  yorum → test.csv


In [25]:
# Bölme sonrası sınıf dağılımı kontrolü
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (bolum_adi, bolum_df) in zip(axes, [
    ("Train", df_train), ("Val", df_val), ("Test", df_test)
]):
    bolum_df[LABEL_COL].value_counts().plot(kind="bar", ax=ax,
        color=renk_paleti, edgecolor="white")
    ax.set_title(f"{bolum_adi} Sınıf Dağılımı")
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "bolme_dagilimi.png"), dpi=150)
plt.close(fig)
print(f"  Grafik kaydedildi: {OUTPUT_DIR}/bolme_dagilimi.png")

  Grafik kaydedildi: eda_rapor/bolme_dagilimi.png


In [26]:
# Class weight hesapla (eğitimde kullanmak için)
sinif_sayilari = df_train["etiket_id"].value_counts().sort_index()
agirliklar     = 1.0 / sinif_sayilari
agirliklar     = agirliklar / agirliklar.sum() * len(ETIKET_MAP)  # normalize
print("\n  Sınıf Ağırlıkları (class_weight için):")
for etiket, agirlik in zip(ETIKET_MAP.keys(), agirliklar.values):
    print(f"    {etiket:<18}: {agirlik:.4f}")


  Sınıf Ağırlıkları (class_weight için):
    olumlu            : 0.0550
    olumsuz           : 0.0933
    nötr              : 0.1287
    öneri/tavsiye     : 0.4891
    şikayet           : 4.2338


In [27]:
# Ağırlıkları JSON olarak kaydet
agirlik_dict = {str(k): float(v) for k, v in zip(sinif_sayilari.index, agirliklar.values)}
with open("class_weights.json", "w", encoding="utf-8") as f:
    json.dump(agirlik_dict, f, ensure_ascii=False, indent=2)
print("  Sınıf ağırlıkları kaydedildi: class_weights.json")

  Sınıf ağırlıkları kaydedildi: class_weights.json


## AŞAMA 4 : TOKENİZATION

In [28]:
print(f"  [4.1] Tokenizer yükleniyor: {MODEL_ADI}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)

  [4.1] Tokenizer yükleniyor: xlm-roberta-base


In [29]:
# 1. Eğer 'kelime_sayisi' sütunun yoksa önce onu hesaplayalım (garanti olsun)
df["kelime_sayisi"] = df[TEXT_COL].apply(lambda x: len(str(x).split()))

# 2.80 kelimeden uzun olanları filtrele
uzun_yorumlar = df[df["kelime_sayisi"] >80]

# 3. İstatistikleri yazdır
toplam_yorum = len(df)
uzun_sayisi = len(uzun_yorumlar)
yuzde = (uzun_sayisi / toplam_yorum) * 100

print(f"--- Uzunluk Analizi (>80 Kelime) ---")
print(f"Toplam Yorum Sayısı         : {toplam_yorum}")
print(f"84 Kelimeden Uzun Yorumlar  : {uzun_sayisi}")
print(f"Veri Setine Oranı           : %{yuzde:.2f}")

# 4. Bu uzun yorumlardan 3 tane örnek görelim (Neyi kırpıyoruz?)
if uzun_sayisi > 0:
    print("\n--- Uzun Yorum Örnekleri (İlk 3) ---")
    for i, yorum in enumerate(uzun_yorumlar[TEXT_COL].head(3)):
        print(f"{i+1}. [Kelime: {len(yorum.split())}] {yorum[:150]}...")

MAX_LENGTH =80  # modelin maksimum token sınırını bu şekilde düzenledik EDA da gördüğümüz üzere ve burada sayısal 
# olarak ispatladığımız üzere 80 kelime üzeri olan yorumlar sadece 51 adet bu yüzden bu şekilde kısıtlıyoruz 

--- Uzunluk Analizi (>80 Kelime) ---
Toplam Yorum Sayısı         : 7212
84 Kelimeden Uzun Yorumlar  : 51
Veri Setine Oranı           : %0.71

--- Uzun Yorum Örnekleri (İlk 3) ---
1. [Kelime: 96] İlk kez yorum yapacağım galiba Bu zorlayıcı yks serüvenimde kafa dağıtmak ve bir nebze olsun gülüp eğlenmek çok kıymetli oluyor Bu vakti Açık Mutfak s...
2. [Kelime: 87] Uzun zamandır bir videoyu bu kadar eğlenerek izlemedim Açık mutfağı tesadüfi bir şekilde yeni keşfettim Ve ilk defa bir içeriğin altına yorum yapıyoru...
3. [Kelime: 91] Emre abi seni çok seviyorum tabi İnci yi de Söyleyeceğim bir şey var Emre abi sizi o kadar çok seviyorum ki rüyama bile girdiniz İsterseniz söyliyim b...


In [30]:
# ── 4.2 TOKENIZE FONKSİYONU ──────────────────────────────────
def tokenize_veri(metin_listesi, max_length=MAX_LENGTH):
    """
    Padding   : kısa cümlelerin sonuna [PAD] token'ı ekle
    Truncation: max_length üstü cümleleri kırp
    Attention Mask: gerçek token=1, padding token=0
    """
    return tokenizer(
        metin_listesi,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",    # PyTorch tensörü döndür
        return_attention_mask=True,
    )

In [31]:
# --- DataFrame Tokenize ve Etiketleme ---
def df_tokenize_et(bolum_df, desc):
    metinler = bolum_df[TEXT_COL].tolist()
    etiketler = torch.tensor(bolum_df["etiket_id"].tolist(), dtype=torch.long)
    
    TOKENIZE_BATCH = 512
    input_ids_list      = []
    attention_mask_list = []

    for i in tqdm(range(0, len(metinler), TOKENIZE_BATCH), desc=f"  {desc} Tokenize"):
        batch = metinler[i: i + TOKENIZE_BATCH]
        enc   = tokenize_veri(batch)
        input_ids_list.append(enc["input_ids"])
        attention_mask_list.append(enc["attention_mask"])

    return {
        "input_ids": torch.cat(input_ids_list, dim=0),
        "attention_mask": torch.cat(attention_mask_list, dim=0),
        "labels": etiketler
    }

In [32]:
# ── 4.3 TÜM SETLERİ TOKENİZE ET ─────────────────────────────
print(f"  [4.3] Veri setleri tokenize ediliyor (max_length={MAX_LENGTH})...")

train_tensorlar = df_tokenize_et(df_train, "Train")
val_tensorlar   = df_tokenize_et(df_val,   "Val  ")
test_tensorlar  = df_tokenize_et(df_test,  "Test ")

print(f"  Train input_ids shape : {train_tensorlar['input_ids'].shape}")
print(f"  Val input_ids shape   : {val_tensorlar['input_ids'].shape}")

  [4.3] Veri setleri tokenize ediliyor (max_length=80)...


  Test  Tokenize: 100%|██████████| 2/2 [00:00<00:00, 33.75it/s]

  Train input_ids shape : torch.Size([5769, 80])
  Val input_ids shape   : torch.Size([721, 80])


In [33]:
# ── 4.4 TENSÖRLERI TEK DOSYADA KAYDET ───────────────────────
print(f"  [4.4] Tüm tensörler kaydediliyor: {TOKEN_DOSYA}")
torch.save({
    "train": train_tensorlar,
    "val": val_tensorlar,
    "test": test_tensorlar
}, TOKEN_DOSYA)

  [4.4] Tüm tensörler kaydediliyor: tokenized.pt


In [34]:
# ── 4.5 ÖRNEK TOKENIZASYON KONTROLÜ ──────────────────────────
print("\n  [4.5] Örnek tokenizasyon kontrolü (Train 0. index):")
ornek_metin = df_train[TEXT_COL].iloc[0]
ornek_tokenlar = tokenizer.convert_ids_to_tokens(train_tensorlar["input_ids"][0])
print(f"  Metin   : {ornek_metin[:80]}")
print(f"  Tokenlar: {ornek_tokenlar[:20]}")
print(f"  Gerçek token sayısı (mask=1): {train_tensorlar['attention_mask'][0].sum().item()}")


  [4.5] Örnek tokenizasyon kontrolü (Train 0. index):
  Metin   : ay cok mutlu oldum 🎉❤
  Tokenlar: ['<s>', '▁ay', '▁cok', '▁mutlu', '▁oldum', '▁', '🎉', '❤', '</s>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
  Gerçek token sayısı (mask=1): 9


In [35]:
# ──────────────────────────────────────────────────────────────
# ÖZET
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PIPELINE TAMAMLANDI")
print("=" * 60)
print(f"  Temiz veri          : {TEMIZ_DOSYA}  ({len(df_temiz)} yorum)")
print(f"  Train               : {TRAIN_DOSYA}  ({len(df_train)} yorum)")
print(f"  Validation          : {VAL_DOSYA}   ({len(df_val)} yorum)")
print(f"  Test                : {TEST_DOSYA}   ({len(df_test)} yorum)")
print(f"  Tokenized           : {TOKEN_DOSYA}")
print(f"  Class weights       : class_weights.json")
print(f"  EDA grafikleri      : {OUTPUT_DIR}/")
print("=" * 60)
print("\nSonraki adım: fine_tune.py ile modeli eğit")


PIPELINE TAMAMLANDI
  Temiz veri          : temiz_veri.csv  (7212 yorum)
  Train               : train.csv  (5769 yorum)
  Validation          : val.csv   (721 yorum)
  Test                : test.csv   (722 yorum)
  Tokenized           : tokenized.pt
  Class weights       : class_weights.json
  EDA grafikleri      : eda_rapor/

Sonraki adım: fine_tune.py ile modeli eğit
